# Agent for S05E01: Safe Code Execution

This notebook implements an agent that safely executes code with multiple security layers as discussed in the S05E01 lesson.

## Security Layers:
1. Static Analysis
2. Sandboxing
3. Output Validation
4. Human Confirmation

In [ ]:
# Install required packages
!pip install bandit e2b-code-interpreter

In [ ]:
import ast
import subprocess
import shlex
from e2b_code_interpreter import CodeInterpreter
import bandit

class SafeCodeExecutor:
    def __init__(self):
        self.sandbox = CodeInterpreter()
    
    def static_analysis(self, code):
        # Use bandit for security analysis
        try:
            b_mgr = bandit.BanditManager(config_file=None, agg_type='file')
            b_mgr.run_tests([code])
            issues = b_mgr.get_results()
            return len(issues) == 0, issues
        except:
            return True, []  # Assume safe if analysis fails
    
    def validate_output(self, output):
        # Check for injection patterns
        dangerous_patterns = ['import os', 'subprocess', 'eval', 'exec']
        for pattern in dangerous_patterns:
            if pattern in output:
                return False
        return True
    
    def human_confirmation(self, code):
        print(f"Code to execute:\n{code}")
        response = input("Approve execution? (yes/no): ")
        return response.lower() == 'yes'
    
    def execute(self, code):
        # Layer 1: Static Analysis
        safe, issues = self.static_analysis(code)
        if not safe:
            return f"Security issues found: {issues}"
        
        # Layer 2: Human Confirmation
        if not self.human_confirmation(code):
            return "Execution cancelled by user"
        
        # Layer 3: Sandboxed Execution
        try:
            result = self.sandbox.run(code)
            output = result.output
            
            # Layer 4: Output Validation
            if not self.validate_output(output):
                return "Output contains dangerous content"
            
            return output
        except Exception as e:
            return f"Execution error: {str(e)}"

executor = SafeCodeExecutor()

In [ ]:
# Example: Safe code
safe_code = """
print('Hello, World!')
result = 2 + 3
print(result)
"""

print("Executing safe code:")
print(executor.execute(safe_code))

In [ ]:
# Example: Potentially unsafe code
unsafe_code = """
import os
os.system('ls')
"""

print("Executing unsafe code:")
print(executor.execute(unsafe_code))